# Imort packages

In [ ]:
import pandas as pd
import numpy as np

print("Kernel is working!")

# 1. Load the C-MAPss training file

In [ ]:
import pandas as pd
import numpy as np

column_names = (
    ['unit', 'cycle']
    + [f'op_setting_{i}' for i in range(1, 4)]
    + [f'sensor_{i}' for i in range(1, 22)]
)

train_df = pd.read_csv(
    '../data/raw/train_FD001.txt',
    sep=r'\s+',
    header=None,
    names=column_names
)

train_df.head()

# 2. Check the shape and basic structure

In [ ]:
train_df.shape

In [ ]:
train_df.info()

In [ ]:
train_df.describe()

# 3. Check for missing values

In [ ]:
train_df.isnull().sum()

# 4. Check how many engine units are in FD001

In [ ]:
train_df['unit'].nunique()

# 5. Plot one sensor for one engine

In [ ]:
import matplotlib.pyplot as plt

engine_1 = train_df[train_df['unit'] == 1]

plt.figure(figsize=(10, 4))
plt.plot(engine_1['cycle'], engine_1['sensor_2'])
plt.xlabel('Cycle')
plt.ylabel('Sensor 2')
plt.title('Sensor 2 over Time for Engine 1')
plt.show()

# 6) Checking maximum cycle per engine

In [ ]:
max_cycles = train_df.groupby('unit')['cycle'].max()
max_cycles.head()

In [ ]:
max_cycles.describe()

# 7) Identify constant columns

   from the research we know that there are some sensors in the C-MAPSS data that are known to be less informative. Here we check for variance.

In [ ]:
train_df.nunique()

In [ ]:
constant_columns = [col for col in train_df.columns if train_df[col].nunique() == 1]
constant_columns

Shows constant columns - we will remove these from the dataset as they contain very little useful information therefor hurting the model learning and there is a lot of literature to support this decision 

# 8) Checking for low-variance sensors

Some columns might not be fully constant, but barely change.

In [ ]:
variance_df = train_df.var(numeric_only=True).sort_values()
variance_df

# Zero or Near-Zero Variance

These are clearly constant or effectively constant:

- sensor_1        0
- op_setting_3    0
- sensor_10       0
- sensor_18       0
- sensor_19       0
- sensor_16       ~1e-35
- sensor_5        ~1e-29

Remove these sensors

# Very Low Variance

Borderline - I will keep them for now and re-evaluate later

- op_setting_2    ~1e-8
- sensor_6        ~1e-6
- op_setting_1    ~1e-6

I do not want to "over prune" the data set just yet, some literature keep these sensors

# Low–Moderate Variance

These are useful but subtle

- sensor_15
- sensor_8
- sensor_13
- sensor_21
- sensor_20
- sensor_11
- sensor_2
- sensor_12
- sensor_7
- sensor_17

These could be valid sensors, possibly just giving weaker indications of failure

# High Variance

possibly the most informative sensors

- sensor_3   ~37
- sensor_4   ~81
- sensor_14  ~363
- sensor_9   ~487

High variance doesn't always mean best feature

In this dataset, high variance sensors tend to correlate with degredation. The literature also confirms that sensors such as the following, are highly informative. 

- sensor_4   ~81
- sensor_14  ~363
- sensor_9   ~487

# 9) Visualising sensors for one engine

Looking for sensors that show degredation trends

In [ ]:
sensors_to_plot = ['sensor_2', 'sensor_3', 'sensor_4', 'sensor_7', 'sensor_11', 'sensor_12', 'sensor_15']

engine_1 = train_df[train_df['unit'] == 1]

for sensor in sensors_to_plot:
    plt.figure(figsize=(10, 4))
    plt.plot(engine_1['cycle'], engine_1[sensor])
    plt.xlabel('Cycle')
    plt.ylabel(sensor)
    plt.title(f'{sensor} over Time for Engine 1')
    plt.show()

# 10) Adding RUL labels to the training set

for the training data, RUL is = maximum cycle - current cycle (for one engine)

In [ ]:
rul = train_df.groupby('unit')['cycle'].max().reset_index()
rul.columns = ['unit', 'max_cycle']

train_df = train_df.merge(rul, on='unit', how='left')
train_df['RUL'] = train_df['max_cycle'] - train_df['cycle']

train_df[['unit', 'cycle', 'max_cycle', 'RUL']].head()

In [ ]:
train_df[['RUL']].describe()

# 11) Plotting RUL for one engine

In [ ]:
engine_1 = train_df[train_df['unit'] == 1]

plt.figure(figsize=(10, 4))
plt.plot(engine_1['cycle'], engine_1['RUL'])
plt.xlabel('Cycle')
plt.ylabel('RUL')
plt.title('RUL over Time for Engine 1')
plt.show()

# 12) Piecewise RUL capping

Capping is utilised as early life RUL is very large (e.g. 200+) and models tend to focus too much on large values therefore making prediction harder.

So we will cap it at 125 cycles.

In [ ]:
RUL_CAP = 125

train_df['RUL_capped'] = train_df['RUL'].clip(upper=RUL_CAP)

train_df[['RUL', 'RUL_capped']].head(10)

In [ ]:
engine_1 = train_df[train_df['unit'] == 1]

plt.figure(figsize=(10, 4))
plt.plot(engine_1['cycle'], engine_1['RUL'], label='Original RUL')
plt.plot(engine_1['cycle'], engine_1['RUL_capped'], label='Capped RUL')
plt.xlabel('Cycle')
plt.ylabel('RUL')
plt.title('RUL vs Capped RUL')
plt.legend()
plt.show()